### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [2]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], dtype=int64)

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [26]:
# PUNTO 1: Vectorizar Documentos

import pandas as pd

# Configuramos el escenario SIN LIMPIEZA (Vectorizador original)
X_tr_raw = X_train
vocab_raw = tfidfvect.vocabulary_
idx2word_raw = idx2word
count_vec_raw = CountVectorizer(vocabulary=vocab_raw)

# Configuramos el escenario CON LIMPIEZA (Stop words + poda de extremos)
vec_clean = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.85)
X_tr_clean = vec_clean.fit_transform(newsgroups_train.data)
vocab_clean = vec_clean.vocabulary_
idx2word_clean = {v: k for k, v in vocab_clean.items()}
count_vec_clean = CountVectorizer(vocabulary=vocab_clean)

print(f"Vocabulario SIN limpieza: {X_tr_raw.shape[1]} términos")
print(f"Vocabulario CON limpieza: {X_tr_clean.shape[1]} términos\n")

# Función para generar e imprimir tablas de resultados dado un escenario
def analizar_escenario(idx_base, X_matriz, count_vec, dic_idx2word, nombre_escenario):
    print(f"--- ESCENARIO: {nombre_escenario} ---")
    
    # Similitud coseno contra todos los documentos de train
    cossim = cosine_similarity(X_matriz[idx_base], X_matriz)[0]
    most_sim_idx = np.argsort(cossim)[::-1][1:6]
    most_sim_val = np.sort(cossim)[::-1][1:6]
    
    # Tabla 1: Top 5 documentos más similares
    filas_similares = []
    aciertos = 0
    for rank, (sim_idx, sim_val) in enumerate(zip(most_sim_idx, most_sim_val), 1):
        clase_sim = newsgroups_train.target_names[y_train[sim_idx]]
        coincide = (y_train[sim_idx] == y_train[idx_base])
        aciertos += int(coincide)
        filas_similares.append({
            "Rank": rank,
            "Idx Doc": sim_idx,
            "Similitud Coseno": round(sim_val, 4),
            "Clase": clase_sim,
            "Coincidencia": "MISMA CLASE" if coincide else "DISTINTA CLASE"
        })
        
    df_similares = pd.DataFrame(filas_similares)
    print(f"\n>>> TOP 5 DOCUMENTOS MÁS SIMILARES [{nombre_escenario}] (Coincidencias: {aciertos}/5):")
    print(df_similares.to_string(index=False))
    
    # Tabla 2: Top 10 palabras repetidas en los 6 documentos (base + 5 similares)
    seis_docs_idx = np.concatenate(([idx_base], most_sim_idx))
    textos_6_docs = [newsgroups_train.data[d_idx] for d_idx in seis_docs_idx]
    
    mat_conteos_6 = count_vec.transform(textos_6_docs).toarray()
    
    # Palabras que aparecen en el documento base y en al menos uno de los 5 similares
    mask_compartidas = (mat_conteos_6[0] > 0) & ((mat_conteos_6[1:] > 0).sum(axis=0) > 0)
    indices_compartidos = np.where(mask_compartidas)[0]
    
    repeticiones_totales_6 = mat_conteos_6.sum(axis=0)
    docs_presentes_6 = (mat_conteos_6 > 0).sum(axis=0)
    
    orden = np.argsort(repeticiones_totales_6[indices_compartidos])[::-1]
    top_palabras_idx = indices_compartidos[orden][:10]
    
    filas_palabras = []
    for rank_w, w_idx in enumerate(top_palabras_idx, 1):
        filas_palabras.append({
            "Rank": rank_w,
            "Palabra": dic_idx2word[w_idx],
            "Repeticiones Totales (en los 6 docs)": repeticiones_totales_6[w_idx],
            "Docs que la contienen": f"{docs_presentes_6[w_idx]}/6"
        })
        
    df_palabras = pd.DataFrame(filas_palabras)
    print(f"\n>>> TOP 10 PALABRAS MÁS REPETIDAS EN LOS 6 DOCUMENTOS [{nombre_escenario}]:")
    print(df_palabras.to_string(index=False))
    print("\n")

# Semilla reproducible: elegimos 5 documentos con al menos 15 palabras
np.random.seed(42)
doc_lengths = np.diff(X_tr_clean.indptr)
valid_docs = np.where(doc_lengths >= 15)[0]
random_indices = np.random.choice(valid_docs, size=5, replace=False)

# Iteramos sobre los 5 documentos mostrando ambos escenarios para cada uno
for i, idx in enumerate(random_indices, 1):
    clase_base = newsgroups_train.target_names[y_train[idx]]
    texto_base = newsgroups_train.data[idx].strip()
    
    print("=" * 90)
    print(f"EXPERIMENTO {i}/5 | DOCUMENTO BASE IDX: {idx} | CLASE: {clase_base}")
    print("-" * 90)
    print(texto_base[:500] + ("\n...[TEXTO TRUNCADO]..." if len(texto_base) > 500 else ""))
    print("=" * 90 + "\n")
    
    # Ejecutamos sin limpieza y con limpieza para el mismo documento
    analizar_escenario(idx, X_tr_raw, count_vec_raw, idx2word_raw, "SIN LIMPIEZA")
    analizar_escenario(idx, X_tr_clean, count_vec_clean, idx2word_clean, "CON LIMPIEZA")

Vocabulario SIN limpieza: 101631 términos
Vocabulario CON limpieza: 39115 términos

EXPERIMENTO 1/5 | DOCUMENTO BASE IDX: 6825 | CLASE: sci.crypt
------------------------------------------------------------------------------------------
And the initiative for actual implementation.  Clinton is not just an
innocent bystander here - They didn't just slap his name on it without
consulting him.  What exactly is his extensive history of individual
rights advocacy that people are assuming he really has nothing to do
with this?  He just went back on his pledge to cut the espionage
budget, and is now asking for more money than even Bush wanted.

--- ESCENARIO: SIN LIMPIEZA ---

>>> TOP 5 DOCUMENTOS MÁS SIMILARES [SIN LIMPIEZA] (Coincidencias: 1/5):
 Rank  Idx Doc  Similitud Coseno                  Clase   Coincidencia
    1     1946            0.2453 soc.religion.christian DISTINTA CLASE
    2    10837            0.2403              sci.crypt    MISMA CLASE
    3     9670            0.2313 soc


>>> TOP 5 DOCUMENTOS MÁS SIMILARES [CON LIMPIEZA] (Coincidencias: 2/5):
 Rank  Idx Doc  Similitud Coseno                 Clase   Coincidencia
    1     9008            0.9716       sci.electronics    MISMA CLASE
    2     5162            0.3853       sci.electronics    MISMA CLASE
    3     4359            0.3366 comp.sys.mac.hardware DISTINTA CLASE
    4     5595            0.3248 comp.sys.mac.hardware DISTINTA CLASE
    5    11290            0.2659 comp.sys.mac.hardware DISTINTA CLASE

>>> TOP 10 PALABRAS MÁS REPETIDAS EN LOS 6 DOCUMENTOS [CON LIMPIEZA]:
 Rank     Palabra  Repeticiones Totales (en los 6 docs) Docs que la contienen
    1         fpu                                    33                   6/6
    2        does                                     9                   5/6
    3         mhz                                     7                   4/6
    4       board                                     6                   3/6
    5        work                             

Al analizar los cinco documentos aleatorios se ve claramente el impacto de filtrar las stop words y podar los extremos del vocabulario, reduciendo las dimensiones de 101.631 a 39.115 términos. Sin limpieza, el ranking de palabras repetidas está totalmente dominado por conectores y artículos como the, to, and u of, lo que introduce ruido y hace que documentos cortos o con vocabulario poco específico se emparejen con textos de cualquier categoría solo por compartir la estructura gramatical. Esto queda muy claro en el tercer experimento (soc.religion.christian), donde sin limpiar no acierta ninguna clase (0/5) y al limpiar pasa a acertar 2/5 gracias a capturar el término específico essene, o en el quinto (rec.autos), donde los aciertos suben de 3/5 a 4/5 al dejar que traccionen palabras propias del problema mecánico como water, trunk, drain y sensor.

Por otro lado, los resultados muestran tanto el punto fuerte de TF-IDF como su principal limitación al trabajar como una bolsa de palabras sin contexto. Cuando el texto tiene jerga técnica muy marcada, como en el segundo documento (comp.sys.ibm.pc.hardware), el método acierta 5/5 en ambos escenarios porque términos como harddisk, eisa, controller y motherboard tienen un peso tan alto que compensan el ruido. Sin embargo, cuando el vocabulario se solapa entre tópicos o usa siglas ambiguas, la similitud coseno se confunde de etiqueta aunque el emparejamiento tenga sentido desde las palabras, como ocurre en el primer, tercer y cuarto caso.

In [28]:
# PUNTO 2: Construir un modelo de clasificación por prototipos (tipo zero-shot).

from sklearn.metrics import confusion_matrix

# Calculamos similitud coseno entre todos los docs de test y todos los de train
sim_matrix = cosine_similarity(X_test, X_train)

# Para cada doc de test, buscamos el índice del doc de train con mayor similitud
idx_mas_similar = np.argmax(sim_matrix, axis=1)
max_sim_val = np.max(sim_matrix, axis=1)

# Asignamos la clase del documento de entrenamiento más cercano
y_pred_zeroshot = y_train[idx_mas_similar]

# Calculamos F1-Score Macro global y por clase
f1_zeroshot = f1_score(y_test, y_pred_zeroshot, average='macro')
f1_por_clase = f1_score(y_test, y_pred_zeroshot, average=None)
docs_sin_match = np.sum(max_sim_val == 0)

# Tabla 1: Comparativa general
df_comparativa = pd.DataFrame([
    {"Modelo": "Clasificador Aleatorio (Azar 1/20)", "F1-Score Macro": 0.0500},
    {"Modelo": "Clasificador por Prototipos (Zero-Shot)", "F1-Score Macro": round(f1_zeroshot, 4)},
    {"Modelo": "Baseline MultinomialNB (Cátedra)", "F1-Score Macro": 0.5854}
])

print(f"Documentos de test con similitud 0 contra todo train: {docs_sin_match}\n")
print(">>> COMPARATIVA GENERAL DE DESEMPEÑO:")
print(df_comparativa.to_string(index=False))

# Tabla 2: Clases mejor y peor clasificadas
df_clases = pd.DataFrame({
    "Clase": newsgroups_test.target_names,
    "F1-Score": np.round(f1_por_clase, 4)
}).sort_values(by="F1-Score", ascending=False)

print("\n>>> TOP 5 CLASES MEJOR CLASIFICADAS:")
print(df_clases.head(5).to_string(index=False))

print("\n>>> TOP 5 CLASES CON PEOR DESEMPEÑO:")
print(df_clases.tail(5).to_string(index=False))

# Tabla 3: Top confusiones entre clases
cm = confusion_matrix(y_test, y_pred_zeroshot)
np.fill_diagonal(cm, 0)

pares_error = []
for i in range(len(newsgroups_test.target_names)):
    for j in range(len(newsgroups_test.target_names)):
        if cm[i, j] > 0:
            pares_error.append({
                "Clase Real": newsgroups_test.target_names[i],
                "Predicho como": newsgroups_test.target_names[j],
                "Errores": cm[i, j]
            })

df_errores = pd.DataFrame(pares_error).sort_values(by="Errores", ascending=False).head(8)
print("\n>>> TOP 8 CONFUSIONES MÁS FRECUENTES ENTRE CLASES:")
print(df_errores.to_string(index=False))

Documentos de test con similitud 0 contra todo train: 224

>>> COMPARATIVA GENERAL DE DESEMPEÑO:
                                 Modelo  F1-Score Macro
     Clasificador Aleatorio (Azar 1/20)          0.0500
Clasificador por Prototipos (Zero-Shot)          0.5050
       Baseline MultinomialNB (Cátedra)          0.5854

>>> TOP 5 CLASES MEJOR CLASIFICADAS:
             Clase  F1-Score
  rec.sport.hockey    0.7347
    comp.windows.x    0.6420
rec.sport.baseball    0.5860
         sci.crypt    0.5697
   rec.motorcycles    0.5687

>>> TOP 5 CLASES CON PEOR DESEMPEÑO:
                Clase  F1-Score
talk.politics.mideast    0.4531
          alt.atheism    0.4252
      sci.electronics    0.4057
   talk.politics.misc    0.3072
   talk.religion.misc    0.2772

>>> TOP 8 CONFUSIONES MÁS FRECUENTES ENTRE CLASES:
              Clase Real            Predicho como  Errores
  soc.religion.christian              alt.atheism       67
      talk.politics.misc       talk.politics.guns       65
      ta

El clasificador por prototipos alcanza un F1-Score Macro de 0.5050 sobre el conjunto de test, superando por diez veces lo que daría una asignación aleatoria entre las 20 categorías (0.0500) y demostrando que medir únicamente la dirección de los vectores TF-IDF con similitud coseno ya captura gran parte de la temática sin necesidad de entrenar parámetros. Sin embargo, queda unos 8 puntos por debajo del baseline de Multinomial Naive Bayes (0.5854) porque al decidir la clase mirando un único documento vecino es muy sensible al ruido individual: alcanza con que un texto de test comparta un par de palabras poco frecuentes o muchas stop words con un documento de entrenamiento para equivocarse, mientras que Naive Bayes consolida las probabilidades de las palabras a lo largo de toda la clase. Además, al haber eliminado cabeceras y citas en la carga de datos, quedaron 224 documentos de test vacíos o sin palabras en común con el vocabulario de entrenamiento (similitud 0 contra todo el corpus), donde argmax asigna por defecto la categoría del primer documento de la matriz (que es rec.autos).

Al analizar el desempeño por categoría, el modelo funciona muy bien en tópicos con jerga cerrada y nombres propios específicos como rec.sport.hockey (0.7347), comp.windows.x (0.6420) o rec.sport.baseball (0.5860), donde los términos técnicos no se pisan con otros foros. En cambio, el rendimiento cae fuerte en categorías más abiertas o discursivas como talk.religion.misc (0.2772) y talk.politics.misc (0.3072). Viendo los pares con más errores, se nota que el clasificador no falla asignando temas al azar, sino que se confunde dentro de la misma familia temática: mezcla discusiones de religión entre soc.religion.christian, alt.atheism y talk.religion.misc, cruza debates políticos entre talk.politics.misc, talk.politics.guns y talk.politics.mideast, y confunde consultas de computación entre comp.sys.mac.hardware y comp.sys.ibm.pc.hardware, ya que al trabajar como bolsa de palabras no puede distinguir la postura del autor ni el contexto cuando dos subforos usan exactamente el mismo vocabulario.

In [35]:
# PUNTO 3: Optimización de modelos Naïve Bayes (Tabla limpia y estructurada)

from IPython.display import display

# Grilla ajustada a lo visto en la teórica y la consigna oficial
experimentos = [
    ("TfidfVectorizer (Baseline)", "No", "MultinomialNB", 1.0, 
     TfidfVectorizer(), MultinomialNB(alpha=1.0)),
     
    ("CountVectorizer", "No", "MultinomialNB", 1.0, 
     CountVectorizer(), MultinomialNB(alpha=1.0)),
     
    ("CountVectorizer", "english", "MultinomialNB", 1.0, 
     CountVectorizer(stop_words='english'), MultinomialNB(alpha=1.0)),
     
    ("TfidfVectorizer", "english", "MultinomialNB", 1.0, 
     TfidfVectorizer(stop_words='english'), MultinomialNB(alpha=1.0)),
     
    ("TfidfVectorizer", "No", "ComplementNB", 1.0, 
     TfidfVectorizer(), ComplementNB(alpha=1.0)),
     
    ("TfidfVectorizer", "english", "ComplementNB", 1.0, 
     TfidfVectorizer(stop_words='english'), ComplementNB(alpha=1.0)),
     
    ("TfidfVectorizer", "english", "MultinomialNB", 0.1, 
     TfidfVectorizer(stop_words='english'), MultinomialNB(alpha=0.1)),
     
    ("TfidfVectorizer", "english", "ComplementNB", 0.1, 
     TfidfVectorizer(stop_words='english'), ComplementNB(alpha=0.1)),
     
    ("TfidfVectorizer", "english", "ComplementNB", 0.5, 
     TfidfVectorizer(stop_words='english'), ComplementNB(alpha=0.5))
]

resultados = []
mejor_f1 = -1
mejor_y_pred = None
baseline_y_pred = None

for vec_nom, sw, mod_nom, alpha, vectorizador, modelo in experimentos:
    X_tr = vectorizador.fit_transform(newsgroups_train.data)
    X_te = vectorizador.transform(newsgroups_test.data)
    
    modelo.fit(X_tr, y_train)
    y_pred_exp = modelo.predict(X_te)
    f1_macro = f1_score(y_test, y_pred_exp, average='macro')
    
    if vec_nom == "TfidfVectorizer (Baseline)" and alpha == 1.0 and sw == "No":
        baseline_y_pred = y_pred_exp
        
    if f1_macro > mejor_f1:
        mejor_f1 = f1_macro
        mejor_y_pred = y_pred_exp
        
    resultados.append({
        "Vectorizador": vec_nom,
        "Stop Words": sw,
        "Clasificador": mod_nom,
        "Alpha": alpha,
        "F1 Macro": round(f1_macro, 4),
        "Mejora vs Base": round(f1_macro - 0.5854, 4)
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="F1 Macro", ascending=False).reset_index(drop=True)
print(">>> COMPARATIVA DE EXPERIMENTOS (ENFOQUE CLASE):")
display(df_resultados)

# Salto por clase respecto al baseline original
f1_base_clases = f1_score(y_test, baseline_y_pred, average=None)
f1_mejor_clases = f1_score(y_test, mejor_y_pred, average=None)

df_salto_clases = pd.DataFrame({
    "Clase": newsgroups_test.target_names,
    "F1 Baseline": np.round(f1_base_clases, 4),
    "F1 Mejor Modelo": np.round(f1_mejor_clases, 4),
    "Salto F1": np.round(f1_mejor_clases - f1_base_clases, 4)
}).sort_values(by="Salto F1", ascending=False).reset_index(drop=True)

print("\n>>> TOP 6 CLASES QUE MÁS MEJORARON RESPECTO AL BASELINE:")
display(df_salto_clases.head(6))

>>> COMPARATIVA DE EXPERIMENTOS (ENFOQUE CLASE):


,Vectorizador,Stop Words,Clasificador,Alpha,F1 Macro,Mejora vs Base
0,TfidfVectorizer,english,ComplementNB,0.5,0.6978,0.1124
1,TfidfVectorizer,english,ComplementNB,1.0,0.6936,0.1082
2,TfidfVectorizer,No,ComplementNB,1.0,0.6930,0.1076
3,TfidfVectorizer,english,ComplementNB,0.1,0.6919,0.1065
4,TfidfVectorizer,english,MultinomialNB,0.1,0.6726,0.0872
5,TfidfVectorizer,english,MultinomialNB,1.0,0.6468,0.0614
6,CountVectorizer,english,MultinomialNB,1.0,0.5968,0.0114
7,TfidfVectorizer (Baseline),No,MultinomialNB,1.0,0.5854,0.0000
8,CountVectorizer,No,MultinomialNB,1.0,0.5121,-0.0733



>>> TOP 6 CLASES QUE MÁS MEJORARON RESPECTO AL BASELINE:


,Clase,F1 Baseline,F1 Mejor Modelo,Salto F1
0,talk.politics.misc,0.1466,0.5166,0.3700
1,soc.religion.christian,0.3577,0.6901,0.3324
2,alt.atheism,0.1272,0.3660,0.2388
3,talk.religion.misc,0.0079,0.2209,0.2130
4,sci.crypt,0.5885,0.7862,0.1977
5,rec.sport.hockey,0.7139,0.8970,0.1832


La optimización de los modelos Naïve Bayes permitió elevar el desempeño global del baseline original (0.5854) hasta un máximo de 0.6978 con la combinación de TfidfVectorizer, filtrado de stop words en inglés y el clasificador ComplementNB con un suavizado alpha = 0.5, logrando una mejora superior a los 11 puntos porcentuales en el F1-Score Macro.

Los resultados de la tabla comparativa evidencian que el salto de rendimiento no depende de contar frecuencias brutas (ya que CountVectorizer sin limpieza incluso empeora el baseline (0.5121), sino de dos decisiones metodológicas fundamentales abordadas en clase:   

En primer lugar, la incorporación de ComplementNB demostró ser el factor más determinante para corregir las deficiencias del modelo. Al estar diseñado específicamente para mitigar los sesgos en clases desbalanceadas y textos largos, ComplementNB ocupa las primeras cuatro posiciones de la tabla con métricas muy superiores a cualquier variante de MultinomialNB. Esto se refleja marcadamente en el análisis por clase donde las categorías discursivas o con menor representación que arrastraban un desempeño malo en el baseline mostraron mucha mejora en el indicador F1, tales como talk.politics.misc (que pasa de 0.1466 a 0.5166) y talk.religion.misc (que trepa desde 0.0079 hasta 0.2209).   

En segundo lugar, combinar TF-IDF con la limpieza de stop words elimina el ruido gramatical y permite que el modelo jerarquice los términos verdaderamente diferenciadores de cada foro. Esto consolida rendimientos excelentes en dominios altamente especializados como sci.crypt (0.7862) y rec.sport.hockey (0.8970), confirmando que ajustar el tipo de vectorizador, limpiar el texto y utilizar un clasificador complementario optimiza notablemente la capacidad predictiva sobre el enfoque probabilístico estándar.

In [29]:
# PUNTO 4: Transponer matriz y calcular similaridad entre palabras

# Transponemos la matriz de train
X_word_doc = X_train.T.tocsr()

# Elegimos 5 palabras representativas y limpias de distintos foros
palabras_elegidas = ['car', 'windows', 'space', 'god', 'hockey']

for palabra in palabras_elegidas:
    if palabra not in tfidfvect.vocabulary_:
        print(f"La palabra '{palabra}' no está en el vocabulario.")
        continue
        
    word_idx = tfidfvect.vocabulary_[palabra]
    
    # Calculamos la similaridad coseno de esta palabra contra todas las demás del vocabulario
    sim_palabras = cosine_similarity(X_word_doc[word_idx], X_word_doc)[0]
    
    # Tomamos el top 5
    top_words_idx = np.argsort(sim_palabras)[::-1][1:6]
    top_words_val = np.sort(sim_palabras)[::-1][1:6]
    
    print(f"--------------------------------------------------")
    print(f"PALABRA ANALIZADA: '{palabra.upper()}'")
    print(f"--------------------------------------------------")
    
    filas_sim = []
    for rank, (w_idx, w_val) in enumerate(zip(top_words_idx, top_words_val), 1):
        filas_sim.append({
            "Rank": rank,
            "Palabra Similar": idx2word[w_idx],
            "Similitud Coseno": round(w_val, 4)
        })
        
    df_sim = pd.DataFrame(filas_sim)
    display(df_sim)
    print("\n")

--------------------------------------------------
PALABRA ANALIZADA: 'CAR'
--------------------------------------------------


,Rank,Palabra Similar,Similitud Coseno
0,1,cars,0.1797
1,2,criterium,0.1770
2,3,civic,0.1748
3,4,owner,0.1689
4,5,dealer,0.1681




--------------------------------------------------
PALABRA ANALIZADA: 'WINDOWS'
--------------------------------------------------


,Rank,Palabra Similar,Similitud Coseno
0,1,dos,0.3037
1,2,ms,0.2320
2,3,microsoft,0.2219
3,4,nt,0.2140
4,5,for,0.1930




--------------------------------------------------
PALABRA ANALIZADA: 'SPACE'
--------------------------------------------------


,Rank,Palabra Similar,Similitud Coseno
0,1,nasa,0.3304
1,2,seds,0.2966
2,3,shuttle,0.2928
3,4,enfant,0.2803
4,5,seti,0.2465




--------------------------------------------------
PALABRA ANALIZADA: 'GOD'
--------------------------------------------------


,Rank,Palabra Similar,Similitud Coseno
0,1,jesus,0.2688
1,2,bible,0.2616
2,3,that,0.2560
3,4,existence,0.2548
4,5,christ,0.2511




--------------------------------------------------
PALABRA ANALIZADA: 'HOCKEY'
--------------------------------------------------


,Rank,Palabra Similar,Similitud Coseno
0,1,ncaa,0.2743
1,2,nhl,0.2653
2,3,affiliates,0.2480
3,4,xenophobes,0.2426
4,5,sportschannel,0.2228


A partir de los resultados obtenidos al transponer la matriz documento-término (X_train.T), se observa que la similitud coseno entre vectores de palabras logra capturar de manera efectiva relaciones temáticas y contextuales basándose en la co-ocurrencia dentro de los mismos hilos de discusión del corpus. Para términos con dominios muy acotados (como 'SPACE' con nasa, shuttle y seti; 'WINDOWS' con dos, ms y microsoft; o 'HOCKEY' con nhl y ncaa) el modelo funciona muy bien.

No obstante, el método también expone las limitaciones lógicas de trabajar con una matriz de documentos sin un modelo semántico profundo. En algunas listas se filtran términos menos específicos o conectores (como for en el caso de windows o that en god), y asociaciones que dependen puramente del contenido de los debates del dataset en lugar de una relación estrictamente real. En conclusión, el enfoque demuestra ser útil a nivel macro para mapear campos semánticos por proximidad de documentos, aunque resulta sensible al ruido propio del texto plano.